In [1]:

code_content = r'''#!/usr/bin/env python3
from __future__ import annotations
import argparse
import math
import random
import sys
from dataclasses import dataclass, field

TOKENIZER_A = {"vocab": ["order", "agent", "the", "credit", "policy", "late", "1043", "ic", "ing", "ed", " ", "-", "A", "#"], "name": "teach-A"}
TOKENIZER_B = {"vocab": ["order", "ag", "ent", "the", "cred", "it", "pol", "icy", "late", "10", "43", "ic", "ing", "ed", " ", "-", "A", "#"], "name": "teach-B"}

def _greedy_split(text: str, vocab: list[str]) -> list[str]:
    vocab = sorted(vocab, key=len, reverse=True)
    text, out, i = text.lower(), [], 0
    while i < len(text):
        for v in vocab:
            if v and text.startswith(v.lower(), i):
                out.append(v)
                i += len(v)
                break
        else:
            out.append(text[i])
            i += 1
    return out

def count_tokens(text: str, tokenizer: dict) -> int:
    return len(_greedy_split(text, tokenizer["vocab"]))

@dataclass
class ContextPlan:
    kept: list[str] = field(default_factory=list)
    dropped: list[str] = field(default_factory=list)
    rejected: bool = False
    reason: str = ""

def prepare_context(messages: list[str], context_limit: int, reserved_output: int, tokenizer: dict, strategy: str = "drop_oldest") -> ContextPlan:
    input_budget = context_limit - reserved_output
    msg_tokens = [(msg, count_tokens(msg, tokenizer)) for msg in messages]
    total_tokens = sum(t for _, t in msg_tokens)
    if total_tokens <= input_budget: return ContextPlan(kept=messages, dropped=[], rejected=False)
    if strategy == "reject": return ContextPlan(kept=[], dropped=messages, rejected=True, reason="Exceed budget")
    elif strategy == "drop_oldest":
        system_msg, system_cost = messages[0], count_tokens(messages[0], tokenizer)
        current_budget, kept, dropped, temp_kept = input_budget - system_cost, [messages[0]], [], []
        if current_budget < 0: return ContextPlan(kept=[], dropped=messages, rejected=True, reason="System msg too long")
        for msg in reversed(messages[1:]):
            cost = count_tokens(msg, tokenizer)
            if current_budget >= cost: current_budget -= cost; temp_kept.insert(0, msg)
            else: dropped.append(msg)
        return ContextPlan(kept=kept + temp_kept, dropped=dropped, rejected=False)
    return ContextPlan(rejected=True, reason="Unknown strategy")

def sample_next(distribution: dict[str, float], temperature: float, rng: random.Random) -> str:
    if not distribution: raise ValueError("Empty")
    if temperature <= 0: return sorted(distribution.items(), key=lambda x: (-x[1], x[0]))[0][0]
    tokens, probs = list(distribution.keys()), list(distribution.values())
    scaled_logits = [math.log(p)/temperature if p > 0 else -float('inf') for p in probs]
    max_logit = max(scaled_logits)
    exp_logits = [math.exp(l - max_logit) for l in scaled_logits]
    sum_exp = sum(exp_logits)
    norm = [e/sum_exp for e in exp_logits] if sum_exp > 0 else [1.0/len(tokens)] * len(tokens)
    r, cumulative = rng.random(), 0.0
    for token, p in zip(tokens, norm):
        cumulative += p
        if r <= cumulative: return token
    return tokens[-1]

def _run_self_test() -> int:
    failures = []
    try:
        assert count_tokens("order A-1043", TOKENIZER_A) > 0
        msgs = ["system", "one", "two"]
        drop = prepare_context(msgs, 40, 4, TOKENIZER_A, "drop_oldest")
        assert drop.kept and drop.kept[0] == "system"
        dist = {"Paris": 0.82, "London": 0.11}
        assert sample_next(dist, 0.0, random.Random(1)) == "Paris"
    except Exception as e: failures.append(str(e))
    if failures: print("FAILURES:", failures); return 1
    print("ALL SELF-TESTS PASSED"); return 0

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--self-test", action="store_true")
    args = parser.parse_args()
    if args.self_test: sys.exit(_run_self_test())
'''

with open("COSC726_W02_llm_foundations.py", "w", encoding="utf-8") as f:
    f.write(code_content)

print("تم إنشاء ملف COSC726_W02_llm_foundations.py بنجاح!")

تم إنشاء ملف COSC726_W02_llm_foundations.py بنجاح!


In [2]:

!python COSC726_W02_llm_foundations.py --self-test

ALL SELF-TESTS PASSED
